In [1]:
import pandas as pd
import numpy as np

def run_inventory_optimization_engine(input_path: str, output_path: str, lead_time_days: float = 7.0, target_csl: float = 0.95):
    print("[+] Initiating Module 4: Prescriptive Inventory Optimization Layer...")
    
    # 1. Load Forecast Dataset
    df = pd.read_csv(input_path)
    
    # Map Cycle Service Level (CSL) to Normal Distribution Z-Score
    csl_z_map = {0.90: 1.28, 0.95: 1.65, 0.99: 2.33}
    z_score = csl_z_map.get(target_csl, 1.65)
    
    # 2. Extract Inventory Parameters
    D = df['Forecast_Sales']  # Forecasted daily demand
    sigma_D = df['Sales_Rolling_Std_7']  # Historical daily demand volatility
    L = lead_time_days  # Vendor lead time in days
    sigma_L_time = 1.0  # Vendor lead time variance in days
    
    # 3. Compute Combined Standard Deviation of Lead-Time Demand (sigma_L)
    # Formula: sqrt( L * sigma_D^2 + D^2 * sigma_L_time^2 )
    sigma_L = np.sqrt(L * (sigma_D**2) + (D**2) * (sigma_L_time**2))
    
    # 4. Calculate Safety Stock (SS)
    df['Safety_Stock'] = np.ceil(z_score * sigma_L)
    
    # 5. Calculate Dynamic Reorder Point (ROP)
    df['Reorder_Point'] = np.ceil((D * L) + df['Safety_Stock'])
    
    # 6. Save Decision Schema
    df.to_csv(output_path, index=False)
    
    print(f"[+] Target Cycle Service Level (CSL): {target_csl * 100:.0f}% (Z = {z_score})")
    print(f"[+] Assigned Vendor Lead Time (L): {L} Days")
    print(f"[+] Average Safety Stock Allocation: {df['Safety_Stock'].mean():.2f} Units")
    print(f"[+] Average Dynamic Reorder Trigger (ROP): {df['Reorder_Point'].mean():.2f} Units")
    print(f"[+] Optimized Decision Dataset saved to '{output_path}'. Total Rows: {len(df):,}")
    
    return df

# Execute Module 4 Engine
df_inventory = run_inventory_optimization_engine("rossmann_forecasts.csv", "rossmann_inventory_decisions.csv")

C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


[+] Initiating Module 4: Prescriptive Inventory Optimization Layer...
[+] Target Cycle Service Level (CSL): 95% (Z = 1.65)
[+] Assigned Vendor Lead Time (L): 7.0 Days
[+] Average Safety Stock Allocation: 13115.79 Units
[+] Average Dynamic Reorder Trigger (ROP): 59360.92 Units
[+] Optimized Decision Dataset saved to 'rossmann_inventory_decisions.csv'. Total Rows: 20,582
